In [1]:
from elasticsearch import Elasticsearch
import json
import random

In [2]:
es = Elasticsearch("http://localhost:9200")
print(es.info())

{'name': '15b0c13d7651', 'cluster_name': 'docker-cluster', 'cluster_uuid': 'uJCfvgfmQSaZcJLqd6DsJQ', 'version': {'number': '8.10.2', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': '6d20dd8ce62365be9b1aca96427de4622e970e9e', 'build_date': '2023-09-19T08:16:24.564900370Z', 'build_snapshot': False, 'lucene_version': '9.7.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


In [3]:
res = es.search(
     index="movies_clean",
     query={"match_all": {}}
)

print(json.dumps(res["hits"]["hits"], indent=2))

[
  {
    "_index": "movies_clean",
    "_id": "852892",
    "_score": 1.0,
    "_source": {
      "original_language": "en",
      "popularity": 0.6,
      "id": 852892,
      "production_companies": null,
      "revenue": 0,
      "overview": "Love is in the air as Barney and all his friends gather to exchange cards and celebrate Valentine\u2019s Day. But little does Barney know one of these cards contains a special invitation from the Queen of Hearts herself. She\u2019s inviting everyone to come take a royal tour of Valentine Castle where the rooms are filled with wonderful surprises. There\u2019ll be lots of games songs and sweet treats along the way. And after the children give the Queen her very own card they\u2019ll find Barney has a surprise waiting for them when they get home.",
      "@timestamp": "2026-04-22T07:24:50.469634029Z",
      "vote_count": 0,
      "credits": null,
      "status": "Released",
      "tagline": null,
      "backdrop_path": null,
      "budget": 0,
  

In [8]:
print(es.count(index="movies_raw")["count"])
print(es.count(index="movies_clean")["count"])


1539262
662083


# Contrôle Qualité : movies_raw vs movies_clean

## 1. Nombre de documents

In [4]:
# Comme un df.shape[0] (Le nombre de ligne)
raw_count = es.count(index="movies_raw")["count"]
clean_count = es.count(index="movies_clean")["count"]

print(f"Données Brutes : {raw_count} documents (avec doublons)")
print(f"Données nettoyé : {clean_count} documents")
print(f"Doublons supprimés : {raw_count - clean_count}")

Données Brutes : 1539262 documents (avec doublons)
Données nettoyé : 662083 documents
Doublons supprimés : 877179


## 2. Valeurs manquantes dans movies_clean

In [5]:
# Toute les colonnes
fields = [
    "id", "title", "overview", "tagline", "genres", "original_language",
    "status", "release_date", "popularity", "budget", "revenue", "runtime",
    "vote_average", "vote_count", "production_companies", "credits", "keywords",
    "poster_path", "backdrop_path", "recommendations"
]

# Vérification des valeurs manquantes dans chaque colonne

print(f"{'Champ':<25} {'Manquants':>10} {'%':>8}")
print("-" * 45)
for field in fields:
    count = es.count(index="movies_clean", 
                     body={
                         "query": {
                             "bool": {
                                 "must_not": {
                                     "exists": {
                                         "field": field
                                        }
                                    }
                                }
                            }
                        }
                    )["count"]
    print(f"{field:<25} {count:>10} {count / clean_count * 100:>f}%")

Champ                      Manquants        %
---------------------------------------------
id                                 0 0.000000%
title                              4 0.000604%
overview                      114502 17.294206%
tagline                       568952 85.933637%
genres                        212149 32.042659%
original_language                  0 0.000000%
status                             0 0.000000%
release_date                   59459 8.980596%
popularity                         0 0.000000%
budget                             0 0.000000%
revenue                            0 0.000000%
runtime                        38399 5.799726%
vote_average                       0 0.000000%
vote_count                         0 0.000000%
production_companies          369668 55.834087%
credits                       223092 33.695473%
keywords                      480006 72.499369%
poster_path                   197079 29.766510%
backdrop_path                 480295 72.543020%
recomme

## 3. Comparaison d'un document brut vs nettoyé

In [6]:
# Il prend un document aléatoire 
sample_clean = es.search(index="movies_clean", body={
    "query": {"function_score": {"functions": [{"random_score": {"seed": random.randint(1, 999999), "field": "id"}}]}},
    "size": 1
})
doc = sample_clean["hits"]["hits"][0]["_source"]
doc_id = doc["id"]

# Cherche le même document dans movies_raw pour comparer la donnée brute et celle nettoyée
sample_raw = es.search(index="movies_raw", body={
    "query": {"term": {"id": str(doc_id)}}, "size": 1
})

print("BRUTE (movies_raw)")
if sample_raw["hits"]["hits"]:
    raw_doc = sample_raw["hits"]["hits"][0]["_source"]
    for k in ["title", "release_date", "genres", "budget", "vote_average"]:
        print(f"  {k}: {raw_doc.get(k)}")
else:
    print("  Document non trouvé dans movies_raw")

print()

print("NETTOYÉE (movies_clean)")
for k in ["title", "release_date", "genres", "budget", "vote_average"]:
    print(f"  {k}: {doc.get(k)}")

BRUTE (movies_raw)
  title: Julia
  release_date: 2021-11-04
  genres: Documentary
  budget: 0.0
  vote_average: 4.7

NETTOYÉE (movies_clean)
  title: Julia
  release_date: 2021-11-04T00:00:00.000Z
  genres: ['Documentary']
  budget: 0
  vote_average: 4.7


## 4. Vérification de l'analyzer custom

In [7]:
# On vérifie que l'analyzer custom est bien appliqué sur movies_clean.
# On envoie une phrase test et on observe les tokens produits :
# - "The" supprimé 
# - "Amazing" → "amaz" 
# - "Spider-Man" → "spider", "man" 
# - "returns" → "return" 
result = es.indices.analyze(
    index="movies_clean",
    body={"analyzer": "movies_text_analyzer", "text": "The Amazing Spider-Man returns"}
)
tokens = [t["token"] for t in result["tokens"]]
print("Tokens générés par movies_text_analyzer :")
print(tokens)

Tokens générés par movies_text_analyzer :
['amaz', 'spider', 'man', 'return']
